# 17 — Final MRI-Only Kaggle Inference

**Input:** Kaggle `test.csv`, `test_series.csv`, test DICOM series, and trained NB12/NB13 fold checkpoints.

**Feature extraction:** built into this notebook. It reproduces NB03 preprocessing, NB04 2.5D neighbor construction, and NB09 frozen DINOv2 ViT-S/14 feature extraction for every test slice.

**Predictor:** five-fold hierarchical slice-attention → series-attention → 12-label study classifier.

**Output:** MRI-only study-level probability predictions, binary predictions, per-fold predictions, extraction manifest, and a competition submission only when the provided sample template has an explicit compatible study-level schema.

No radiology reports or ground-truth test labels are consumed.

## 1. Imports, configuration and artifact discovery

Set `MODEL_SOURCE` to `notebook13_weighted` for the planned report-guided final model, or `notebook12_gold_only` for the baseline checkpoint ensemble. The selected checkpoint family must be attached in Kaggle Add Input.

In [1]:
import os, gc, json, random, warnings, glob
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
warnings.filterwarnings("ignore")

# Required by the same DICOM and image preprocessing pipeline used in NB03.
try:
    import pydicom
    import cv2
except ImportError as exc:
    raise ImportError("Notebook 17 requires pydicom and opencv-python, as does Notebook 03.") from exc
try:
    import timm
except ImportError as exc:
    raise ImportError("Notebook 17 requires timm, as does Notebook 09.") from exc

SEED=42; N_FOLDS=5; EMBED_DIM=384; NUM_CLASSES=12; ATTENTION_DIM=128; HIDDEN_DIM=128; DROPOUT=0.25
MODEL_SOURCE="notebook13_weighted"  # options: notebook13_weighted | notebook12_gold_only
WEAK_MODE="gold_plus_confidence_weighted_weak"
THRESHOLD=0.50
SOURCE_IMAGE_SIZE=224
DINO_IMAGE_SIZE=518
DINO_MODEL_NAME="hf_hub:timm/vit_small_patch14_dinov2.lvd142m"
DINO_MODEL_FALLBACK="vit_small_patch14_dinov2.lvd142m"
DINO_BATCH_SIZE=4
USE_AMP=torch.cuda.is_available()
OUTPUT_DIR=Path("/kaggle/working/17_final_inference"); OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
LABEL_COLS=["ACL","MCL","Medial Meniscus","Lateral Meniscus","Medial OA","Lateral OA","PF OA","Effusion","Synovitis","Baker's","Contusion","Fracture"]

def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
seed_everything()
if torch.get_num_threads()>4: torch.set_num_threads(4)

SEARCH_ROOTS=["/kaggle/input","/kaggle/working"]
PRUNE_DIRS={"train_images","test_images","train_series","test_series","processed_data","__pycache__",".git",".ipynb_checkpoints"}

def find_files(filename):
    hits=[]
    for root in SEARCH_ROOTS:
        if not os.path.exists(root): continue
        for current,dirs,files in os.walk(root):
            dirs[:]=[d for d in dirs if d not in PRUNE_DIRS]
            if filename in files: hits.append(os.path.join(current,filename))
    return sorted(set(hits),key=lambda p:(len(p),p))

def choose_file(filename,preferred=(),required=True,override=None):
    if override:
        if not os.path.isfile(override): raise FileNotFoundError(override)
        return override
    hits=find_files(filename)
    if not hits:
        if required: raise FileNotFoundError(f"Cannot find {filename}. Attach the RSNA competition dataset through Kaggle Add Input.")
        return None
    for token in preferred:
        sub=[p for p in hits if token.lower() in p.lower()]
        if sub: hits=sub; break
    if len(hits)>1:
        print(f"Multiple copies of {filename}; selected {hits[0]}")
    return hits[0]

# Use the competition tables only. No labels or reports are consumed in inference.
TEST_CSV_OVERRIDE=None
TEST_SERIES_CSV_OVERRIDE=None
TEST_CSV_PATH=choose_file("test.csv",("rsna-knee-abnormality-detection","rsna"),override=TEST_CSV_OVERRIDE)
TEST_SERIES_CSV_PATH=choose_file("test_series.csv",("rsna-knee-abnormality-detection","rsna"),override=TEST_SERIES_CSV_OVERRIDE)
COMP_DIR=Path(TEST_CSV_PATH).parent
TEST_SERIES_TABLE_DIR=Path(TEST_SERIES_CSV_PATH).parent
SAMPLE_SUB_PATH=choose_file("sample_submission.csv",("rsna-knee-abnormality-detection","rsna"),required=False)

# Locate fold checkpoints produced by NB13 (default) or NB12.
if MODEL_SOURCE=="notebook13_weighted":
    CHECKPOINT_PATHS={f:choose_file(f"{WEAK_MODE}_fold_{f}.pth",("13_weak_supervision","13")) for f in range(N_FOLDS)}
elif MODEL_SOURCE=="notebook12_gold_only":
    CHECKPOINT_PATHS={f:choose_file(f"hierarchical_dinov2_fold_{f}.pth",("12_proposed_model","12")) for f in range(N_FOLDS)}
else:
    raise ValueError("MODEL_SOURCE must be notebook13_weighted or notebook12_gold_only")

print("Competition directory:",COMP_DIR)
print("test.csv:",TEST_CSV_PATH)
print("test_series.csv:",TEST_SERIES_CSV_PATH)
print("Model source:",MODEL_SOURCE,"| device:",DEVICE)
print("Checkpoint paths:",CHECKPOINT_PATHS)
print("Sample submission:",SAMPLE_SUB_PATH or "not attached")


Competition directory: /kaggle/input/competitions/rsna-knee-abnormality-detection
test.csv: /kaggle/input/competitions/rsna-knee-abnormality-detection/test.csv
test_series.csv: /kaggle/input/competitions/rsna-knee-abnormality-detection/test_series.csv
Model source: notebook13_weighted | device: cpu
Checkpoint paths: {0: '/kaggle/input/notebooks/richashanbhag/13-weak-supervision/13_weak_supervision/gold_plus_confidence_weighted_weak_fold_0.pth', 1: '/kaggle/input/notebooks/richashanbhag/13-weak-supervision/13_weak_supervision/gold_plus_confidence_weighted_weak_fold_1.pth', 2: '/kaggle/input/notebooks/richashanbhag/13-weak-supervision/13_weak_supervision/gold_plus_confidence_weighted_weak_fold_2.pth', 3: '/kaggle/input/notebooks/richashanbhag/13-weak-supervision/13_weak_supervision/gold_plus_confidence_weighted_weak_fold_3.pth', 4: '/kaggle/input/notebooks/richashanbhag/13-weak-supervision/13_weak_supervision/gold_plus_confidence_weighted_weak_fold_4.pth'}
Sample submission: /kaggle/inpu

## 2. Build test embeddings directly from Kaggle DICOM inputs

Notebook 09 currently extracts only the gold training studies. This notebook therefore performs the missing test-time feature extraction itself. It follows the supplied notebooks exactly:

- **NB03:** DICOM ordering, per-series 1st/99th percentile clipping, min-max normalization and 224×224 `INTER_AREA` resizing.
- **NB04:** one 2.5D instance per center slice; previous/next indices are clamped at the series boundaries.
- **NB09:** stack channels as previous/current/next, resize to 518×518 with bicubic interpolation, ImageNet normalize, and extract frozen DINOv2 ViT-S/14 384-dimensional features.

No test labels, train labels or radiology reports are read for feature extraction. The two test cache files are written locally inside this notebook and then consumed by the existing inference cells below.

In [2]:
# ------------------------------------------------------------
# 2.1 Validate test tables and locate the test DICOM series
# ------------------------------------------------------------
test_df=pd.read_csv(TEST_CSV_PATH)
test_series_df=pd.read_csv(TEST_SERIES_CSV_PATH)
for col in ["StudyInstanceUID"]:
    if col not in test_df.columns or col not in test_series_df.columns:
        raise ValueError(f"Required column {col} is missing from test tables.")
if "SeriesInstanceUID" not in test_series_df.columns:
    raise ValueError("test_series.csv must contain SeriesInstanceUID.")

test_df["StudyInstanceUID"]=test_df["StudyInstanceUID"].astype(str)
test_series_df["StudyInstanceUID"]=test_series_df["StudyInstanceUID"].astype(str)
test_series_df["SeriesInstanceUID"]=test_series_df["SeriesInstanceUID"].astype(str)
if test_df["StudyInstanceUID"].duplicated().any():
    raise ValueError("test.csv contains duplicate StudyInstanceUID values.")
if test_series_df[["StudyInstanceUID","SeriesInstanceUID"]].duplicated().any():
    raise ValueError("test_series.csv contains duplicate study/series pairs.")

TEST_STUDY_IDS=sorted(test_df["StudyInstanceUID"].dropna().unique().tolist())
series_studies=set(test_series_df["StudyInstanceUID"])
missing_series_studies=set(TEST_STUDY_IDS)-series_studies
extra_series_studies=series_studies-set(TEST_STUDY_IDS)
if missing_series_studies:
    raise ValueError(f"{len(missing_series_studies)} test studies have no rows in test_series.csv; examples: {sorted(missing_series_studies)[:5]}")
if extra_series_studies:
    raise ValueError(f"test_series.csv contains {len(extra_series_studies)} study IDs absent from test.csv.")

# NB03 uses DATA_DIR/train_series/<study>/<series>. Support the corresponding
# test_series path first, and test_images as a fallback for alternate Kaggle mounts.
TEST_SERIES_ROOT_CANDIDATES=[
    COMP_DIR/"test_series",
    TEST_SERIES_TABLE_DIR/"test_series",
    COMP_DIR/"test_images",
    TEST_SERIES_TABLE_DIR/"test_images",
]
TEST_SERIES_ROOTS=[]
for p in TEST_SERIES_ROOT_CANDIDATES:
    if p.is_dir() and p not in TEST_SERIES_ROOTS: TEST_SERIES_ROOTS.append(p)
if not TEST_SERIES_ROOTS:
    raise FileNotFoundError(
        "Could not locate the test DICOM folder. Expected test_series/<StudyInstanceUID>/<SeriesInstanceUID> "
        "next to test.csv (same layout as NB03 train_series), or a test_images equivalent."
    )
print("Test DICOM roots:")
for p in TEST_SERIES_ROOTS: print(" -",p)
print("Test studies:",len(TEST_STUDY_IDS),"| test series rows:",len(test_series_df))

# ------------------------------------------------------------
# 2.2 Reproduce NB03 DICOM ordering and preprocessing exactly
# ------------------------------------------------------------
def safe_get_sort_key(ds, fallback_index):
    try:
        if hasattr(ds,"ImagePositionPatient"):
            return float(ds.ImagePositionPatient[2])
    except Exception:
        pass
    try:
        if hasattr(ds,"InstanceNumber"):
            return float(ds.InstanceNumber)
    except Exception:
        pass
    return float(fallback_index)

def resolve_test_series_dir(study_id,series_id,row=None):
    # First respect any explicit path column that might be present in the series table.
    if row is not None:
        for col in ["File_Path","Series_Path","SeriesPath","path"]:
            if col in row and pd.notna(row[col]):
                raw=Path(str(row[col]))
                if raw.is_dir(): return raw
    for root in TEST_SERIES_ROOTS:
        for candidate in [root/str(study_id)/str(series_id),root/str(series_id)]:
            if candidate.is_dir(): return candidate
    # Last-resort lookup within the already identified test DICOM roots.
    for root in TEST_SERIES_ROOTS:
        for current,dirs,files in os.walk(root):
            dirs[:] = [d for d in dirs if d not in {"__pycache__",".ipynb_checkpoints"}]
            if Path(current).name==str(series_id) and (Path(current).parent.name==str(study_id) or Path(current).parent==root):
                return Path(current)
    raise FileNotFoundError(f"Cannot locate DICOM directory for study={study_id}, series={series_id}.")

def read_dicom_series(series_path):
    dicom_files=[f for f in os.listdir(series_path) if f.lower().endswith(".dcm")]
    if not dicom_files: return None,"No .dcm files found"
    slices=[]
    for i,filename in enumerate(dicom_files):
        filepath=series_path/filename
        try:
            ds=pydicom.dcmread(str(filepath))
            slices.append((safe_get_sort_key(ds,i),ds))
        except Exception as exc:
            return None,f"DICOM read error in {filename}: {exc}"
    if not slices: return None,"No readable DICOM slices"
    slices.sort(key=lambda x:x[0])
    return [ds for _,ds in slices],None

def preprocess_series_like_nb03(series_path,target_size=SOURCE_IMAGE_SIZE):
    slices,err=read_dicom_series(series_path)
    if err is not None: return None,err
    try:
        volume=np.stack([ds.pixel_array for ds in slices]).astype(np.float32)
    except Exception as exc:
        return None,f"Pixel array error: {exc}"
    if volume.ndim!=3: return None,f"Unexpected volume ndim: {volume.ndim}"
    if not np.isfinite(volume).all(): return None,"Non-finite pixel values"
    p1=float(np.percentile(volume,1)); p99=float(np.percentile(volume,99))
    if p99<=p1: return None,"Invalid intensity percentiles"
    volume=np.clip(volume,p1,p99)
    vmin=float(volume.min()); vmax=float(volume.max())
    if vmax<=vmin: return None,"Invalid intensity range after clipping"
    volume=(volume-vmin)/(vmax-vmin)
    resized=np.stack([
        cv2.resize(img,(target_size,target_size),interpolation=cv2.INTER_AREA)
        for img in volume
    ]).astype(np.float16)
    return resized,None

# ------------------------------------------------------------
# 2.3 Reproduce NB04 indexing + NB09 DINOv2 tensor preparation
# ------------------------------------------------------------
IMAGENET_MEAN=torch.tensor([0.485,0.456,0.406],dtype=torch.float32).view(3,1,1)
IMAGENET_STD=torch.tensor([0.229,0.224,0.225],dtype=torch.float32).view(3,1,1)

def make_2p5d_tensor(volume,center_idx):
    n=len(volume)
    prev_idx=max(int(center_idx)-1,0)
    next_idx=min(int(center_idx)+1,n-1)
    x=np.stack([volume[prev_idx],volume[int(center_idx)],volume[next_idx]],axis=0).astype(np.float32)
    x=torch.from_numpy(x)
    x=F.interpolate(x.unsqueeze(0),size=(DINO_IMAGE_SIZE,DINO_IMAGE_SIZE),mode="bicubic",align_corners=False).squeeze(0)
    return (x-IMAGENET_MEAN)/IMAGENET_STD,prev_idx,next_idx

# Load the exact frozen backbone used by NB09.
print("Loading pretrained DINOv2 ViT-S/14:",DINO_MODEL_NAME)
try:
    dinov2=timm.create_model(DINO_MODEL_NAME,pretrained=True,num_classes=0)
except Exception as primary_exc:
    print("Primary timm model name failed; trying NB09 fallback:",repr(primary_exc))
    dinov2=timm.create_model(DINO_MODEL_FALLBACK,pretrained=True,num_classes=0)
dinov2=dinov2.to(DEVICE).eval()
for parameter in dinov2.parameters(): parameter.requires_grad=False

# ------------------------------------------------------------
# 2.4 Extract one embedding for every center slice in every test series
# ------------------------------------------------------------
feature_chunks=[]
metadata_records=[]
series_errors=[]

for series_pos,(_,series_row) in enumerate(test_series_df.iterrows(),start=1):
    study_id=str(series_row["StudyInstanceUID"])
    series_id=str(series_row["SeriesInstanceUID"])
    try:
        series_dir=resolve_test_series_dir(study_id,series_id,series_row)
        volume,err=preprocess_series_like_nb03(series_dir,SOURCE_IMAGE_SIZE)
        if err is not None or volume is None:
            raise RuntimeError(err or "Preprocessing returned no volume")
        n_slices=int(volume.shape[0])
        # NB04: every center slice is retained; boundaries repeat the nearest valid slice.
        series_tensors=[]
        series_rows=[]
        for center_idx in range(n_slices):
            tensor,prev_idx,next_idx=make_2p5d_tensor(volume,center_idx)
            series_tensors.append(tensor)
            rec={
                "StudyInstanceUID":study_id,
                "SeriesInstanceUID":series_id,
                "Anatomical_Plane":series_row.get("Anatomical_Plane",np.nan),
                "Fluid_Sensitive":series_row.get("Fluid_Sensitive",np.nan),
                "Fat_Suppression":series_row.get("Fat_Suppression",np.nan),
                "Num_Slices":n_slices,
                "Prev_Slice_Index":prev_idx,
                "Center_Slice_Index":center_idx,
                "Next_Slice_Index":next_idx,
                "DICOM_Directory":str(series_dir),
            }
            series_rows.append(rec)
        for start in range(0,len(series_tensors),DINO_BATCH_SIZE):
            batch_x=torch.stack(series_tensors[start:start+DINO_BATCH_SIZE]).to(DEVICE,non_blocking=True)
            with torch.inference_mode():
                if USE_AMP:
                    with torch.autocast(device_type="cuda",dtype=torch.float16):
                        batch_features=dinov2(batch_x)
                else:
                    batch_features=dinov2(batch_x)
            batch_features=batch_features.float().cpu().numpy()
            if batch_features.ndim!=2 or batch_features.shape[1]!=EMBED_DIM:
                raise ValueError(f"Expected DINOv2 output [B,{EMBED_DIM}], got {batch_features.shape}")
            feature_chunks.append(batch_features)
            del batch_x,batch_features
        metadata_records.extend(series_rows)
        del volume,series_tensors,series_rows
        if series_pos%10==0 or series_pos==len(test_series_df):
            print(f"Processed test series {series_pos}/{len(test_series_df)}")
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    except Exception as exc:
        series_errors.append({"StudyInstanceUID":study_id,"SeriesInstanceUID":series_id,"Error":str(exc)})
        print(f"ERROR: {study_id}/{series_id}: {exc}")

if series_errors:
    error_path=OUTPUT_DIR/"17_test_feature_extraction_errors.csv"
    pd.DataFrame(series_errors).to_csv(error_path,index=False)
    raise RuntimeError(
        f"Test feature extraction failed for {len(series_errors)} series. "
        f"See {error_path}. No partial predictions will be produced."
    )
if not feature_chunks or not metadata_records:
    raise RuntimeError("No test embeddings were extracted. Check test DICOM paths and test_series.csv.")

test_embeddings=np.concatenate(feature_chunks,axis=0).astype(np.float32)
test_meta=pd.DataFrame(metadata_records)
test_meta.insert(0,"Embedding_Index",np.arange(len(test_meta),dtype=np.int64))
if len(test_embeddings)!=len(test_meta):
    raise RuntimeError("Test embedding/metadata row count mismatch.")
if test_embeddings.shape[1]!=EMBED_DIM:
    raise RuntimeError(f"Expected {EMBED_DIM}-dimensional embeddings, got {test_embeddings.shape}.")
if not np.isfinite(test_embeddings).all():
    raise RuntimeError("Non-finite test embeddings detected.")

TEST_EMBED_PATH=OUTPUT_DIR/"dinov2_vits14_test_embeddings_float16.npy"
TEST_META_PATH=OUTPUT_DIR/"dinov2_vits14_test_embedding_metadata.csv"
TEST_MANIFEST_PATH=OUTPUT_DIR/"dinov2_vits14_test_embedding_manifest.json"
np.save(TEST_EMBED_PATH,test_embeddings.astype(np.float16))
test_meta.to_csv(TEST_META_PATH,index=False)
with open(TEST_MANIFEST_PATH,"w",encoding="utf-8") as f:
    json.dump({
        "model_name":DINO_MODEL_NAME,"embedding_dim":EMBED_DIM,
        "input_size":[3,DINO_IMAGE_SIZE,DINO_IMAGE_SIZE],
        "source_size":[3,SOURCE_IMAGE_SIZE,SOURCE_IMAGE_SIZE],
        "preprocessing_reference":"NB03 preprocess_series; NB04 make_2p5d_index/load_2p5d_sample; NB09 load_2p5d_tensor",
        "slice_sampling":"every center slice, previous/next boundary clamped within series",
        "test_studies":int(test_meta["StudyInstanceUID"].nunique()),
        "test_series":int(test_meta[["StudyInstanceUID","SeriesInstanceUID"]].drop_duplicates().shape[0]),
        "num_embeddings":int(len(test_embeddings)),"dtype":"float16",
        "reports_used":False,"labels_used":False,
    },f,indent=2)

# Downcast to mmap-backed array for hierarchical inference and release extraction buffers.
del feature_chunks,test_embeddings,dinov2
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()
test_embeddings=np.load(TEST_EMBED_PATH,mmap_mode="r")
test_meta=pd.read_csv(TEST_META_PATH)
required_meta={"StudyInstanceUID","SeriesInstanceUID","Embedding_Index"}
missing=required_meta-set(test_meta.columns)
if missing: raise ValueError(f"Generated test metadata missing columns: {sorted(missing)}")
if test_embeddings.ndim!=2 or test_embeddings.shape[1]!=EMBED_DIM: raise ValueError(f"Expected test embeddings [N,{EMBED_DIM}], got {test_embeddings.shape}")
if len(test_meta)!=len(test_embeddings): raise ValueError("Generated test embedding/metadata row count mismatch.")
test_meta["StudyInstanceUID"]=test_meta["StudyInstanceUID"].astype(str)
test_meta["SeriesInstanceUID"]=test_meta["SeriesInstanceUID"].astype(str)
test_meta["Embedding_Index"]=pd.to_numeric(test_meta["Embedding_Index"],errors="raise").astype(int)
test_meta=test_meta.sort_values("Embedding_Index").reset_index(drop=True)
if not np.array_equal(test_meta["Embedding_Index"].to_numpy(),np.arange(len(test_meta))):
    raise ValueError("Generated Embedding_Index must be unique, contiguous and zero-based.")
observed_test_ids=set(test_meta["StudyInstanceUID"])
expected_test_ids=set(TEST_STUDY_IDS)
if observed_test_ids!=expected_test_ids:
    raise ValueError(f"Test study coverage mismatch: missing={len(expected_test_ids-observed_test_ids)}, extra={len(observed_test_ids-expected_test_ids)}")
print("TEST DINOv2 FEATURE EXTRACTION COMPLETE")
print("Saved embeddings:",TEST_EMBED_PATH)
print("Saved metadata:",TEST_META_PATH)
print("Embedding shape:",tuple(test_embeddings.shape))
print("Test studies:",len(TEST_STUDY_IDS),"| test series:",test_meta[["StudyInstanceUID","SeriesInstanceUID"]].drop_duplicates().shape[0])


Test DICOM roots:
 - /kaggle/input/competitions/rsna-knee-abnormality-detection/test_series
Test studies: 3 | test series rows: 15
Loading pretrained DINOv2 ViT-S/14: hf_hub:timm/vit_small_patch14_dinov2.lvd142m


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 88.2MB            

model.safetensors: downloading bytes:           |  0.00B            

Processed test series 10/15
Processed test series 15/15
TEST DINOv2 FEATURE EXTRACTION COMPLETE
Saved embeddings: /kaggle/working/17_final_inference/dinov2_vits14_test_embeddings_float16.npy
Saved metadata: /kaggle/working/17_final_inference/dinov2_vits14_test_embedding_metadata.csv
Embedding shape: (557, 384)
Test studies: 3 | test series: 15


## 3. Build hierarchical test bags: study → series → 2.5D instances

The test embeddings are generated in this notebook using the same 2.5D indexing and DINOv2 preprocessing logic as the training pipeline. The metadata includes a zero-based `Embedding_Index`, `StudyInstanceUID`, `SeriesInstanceUID`, and slice indices. Each series is one bag inside its study.

In [3]:
TEST_STUDY_BAGS={}
for sid,g in test_meta.groupby("StudyInstanceUID",sort=True):
    series_bags=[]
    for series_id,sg in g.groupby("SeriesInstanceUID",sort=True):
        idx=sg.sort_values("Embedding_Index")["Embedding_Index"].to_numpy(dtype=int)
        if len(idx)==0: continue
        series_bags.append({"SeriesInstanceUID":str(series_id),"embedding_indices":idx})
    if not series_bags: raise ValueError(f"No MRI series for test study {sid}")
    TEST_STUDY_BAGS[str(sid)]=series_bags
if set(TEST_STUDY_BAGS)!=set(TEST_STUDY_IDS): raise ValueError("Hierarchical test bags do not cover all test studies.")

def get_test_tensors(study_id):
    return [torch.as_tensor(np.asarray(test_embeddings[s["embedding_indices"]],dtype=np.float32),dtype=torch.float32,device=DEVICE) for s in TEST_STUDY_BAGS[str(study_id)]]

print("Hierarchical test bags ready.")
print("Series per study:",pd.Series({sid:len(bag) for sid,bag in TEST_STUDY_BAGS.items()}).describe().to_dict())

Hierarchical test bags ready.
Series per study: {'count': 3.0, 'mean': 5.0, 'std': 0.0, 'min': 5.0, '25%': 5.0, '50%': 5.0, '75%': 5.0, 'max': 5.0}


## 4. Define the checkpoint-compatible hierarchical MIL models

Notebook 12 and Notebook 13 use the same mathematical hierarchy but different internal attention-layer attribute names. The two class definitions below preserve the respective state-dictionary names so that checkpoints load strictly; no approximate or partial weight loading is used.

In [5]:
# Notebook 12 gold-only checkpoint class: keys attention_v / attention_u / attention_w.
class GatedAttentionPoolNB12(nn.Module):
    def __init__(self,input_dim=EMBED_DIM,attention_dim=ATTENTION_DIM):
        super().__init__(); self.norm=nn.LayerNorm(input_dim); self.attention_v=nn.Linear(input_dim,attention_dim); self.attention_u=nn.Linear(input_dim,attention_dim); self.attention_w=nn.Linear(attention_dim,1,bias=False)
    def forward(self,x):
        z=self.norm(x); g=torch.tanh(self.attention_v(z))*torch.sigmoid(self.attention_u(z)); w=torch.softmax(self.attention_w(g).squeeze(-1),dim=0); return (w.unsqueeze(-1)*z).sum(0),w
class HierarchicalDINOv2MILNB12(nn.Module):
    def __init__(self):
        super().__init__(); self.slice_attention=GatedAttentionPoolNB12(); self.series_attention=GatedAttentionPoolNB12(); self.classifier=nn.Sequential(nn.LayerNorm(EMBED_DIM),nn.Linear(EMBED_DIM,HIDDEN_DIM),nn.GELU(),nn.Dropout(DROPOUT),nn.Linear(HIDDEN_DIM,NUM_CLASSES))
    def forward(self,bags):
        sv=[]
        for x in bags: v,_=self.slice_attention(x); sv.append(v)
        z,_=self.series_attention(torch.stack(sv,dim=0)); return self.classifier(z)

# Notebook 13 weak-supervision checkpoint class: keys v / u / w.
class GatedAttentionPoolNB13(nn.Module):
    def __init__(self,input_dim=EMBED_DIM,attention_dim=ATTENTION_DIM):
        super().__init__(); self.norm=nn.LayerNorm(input_dim); self.v=nn.Linear(input_dim,attention_dim); self.u=nn.Linear(input_dim,attention_dim); self.w=nn.Linear(attention_dim,1,bias=False)
    def forward(self,x):
        z=self.norm(x); scores=self.w(torch.tanh(self.v(z))*torch.sigmoid(self.u(z))).squeeze(-1); weights=torch.softmax(scores,dim=0); return (weights.unsqueeze(-1)*z).sum(0),weights
class HierarchicalDINOv2MILNB13(nn.Module):
    def __init__(self):
        super().__init__(); self.slice_attention=GatedAttentionPoolNB13(); self.series_attention=GatedAttentionPoolNB13(); self.classifier=nn.Sequential(nn.LayerNorm(EMBED_DIM),nn.Linear(EMBED_DIM,HIDDEN_DIM),nn.GELU(),nn.Dropout(DROPOUT),nn.Linear(HIDDEN_DIM,NUM_CLASSES))
    def forward(self,bags):
        sv=[]
        for x in bags: v,_=self.slice_attention(x); sv.append(v)
        z,_=self.series_attention(torch.stack(sv,dim=0)); return self.classifier(z)

print("Checkpoint-compatible classes defined for",MODEL_SOURCE)

Checkpoint-compatible classes defined for notebook13_weighted


## 5. Load five fold checkpoints and generate MRI-only predictions

All five checkpoint models are applied to every test study. Their sigmoid probabilities are averaged for the final ensemble. This is a five-model cross-validation ensemble; each checkpoint was trained on its corresponding outer-training folds, not refit on all 58 gold studies.

In [6]:
fold_prediction_frames=[]
for fold in range(N_FOLDS):
    try: ckpt=torch.load(CHECKPOINT_PATHS[fold],map_location="cpu",weights_only=False)
    except TypeError: ckpt=torch.load(CHECKPOINT_PATHS[fold],map_location="cpu")
    if ckpt.get("label_cols")!=LABEL_COLS: raise ValueError(f"Fold {fold}: checkpoint label order mismatch.")
    if MODEL_SOURCE=="notebook13_weighted":
        if ckpt.get("mode")!=WEAK_MODE: raise ValueError(f"Fold {fold}: expected {WEAK_MODE}, checkpoint mode is {ckpt.get('mode')}")
        model=HierarchicalDINOv2MILNB13().to(DEVICE)
    else:
        model=HierarchicalDINOv2MILNB12().to(DEVICE)
    model.load_state_dict(ckpt["model_state_dict"],strict=True)
    model.eval()
    rows=[]
    for sid in TEST_STUDY_IDS:
        bags=get_test_tensors(sid)
        with torch.no_grad(): prob=torch.sigmoid(model(bags)).cpu().numpy()
        row={"StudyInstanceUID":sid,"fold_model":fold}
        for j,label in enumerate(LABEL_COLS): row[f"{label}_prob"]=float(prob[j])
        rows.append(row)
    fold_df=pd.DataFrame(rows)
    fold_df.to_csv(OUTPUT_DIR/f"17_fold_{fold}_predictions.csv",index=False)
    fold_prediction_frames.append(fold_df)
    print(f"Fold model {fold} complete | studies={len(fold_df)} | checkpoint={CHECKPOINT_PATHS[fold]}")
    del model,ckpt; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

fold_predictions=pd.concat(fold_prediction_frames,ignore_index=True)
prob_cols=[f"{c}_prob" for c in LABEL_COLS]
ensemble=fold_predictions.groupby("StudyInstanceUID")[prob_cols].mean().reindex(TEST_STUDY_IDS)
if ensemble.isna().any().any(): raise ValueError("Missing ensemble probability for at least one test study.")
predictions=ensemble.reset_index()
predictions=predictions.rename(columns={"index":"StudyInstanceUID"})
predictions["StudyInstanceUID"]=TEST_STUDY_IDS
predictions.to_csv(OUTPUT_DIR/"17_mri_only_predictions.csv",index=False)

binary=predictions[["StudyInstanceUID"]].copy()
for label in LABEL_COLS: binary[label]=(predictions[f"{label}_prob"]>=THRESHOLD).astype(int)
binary.to_csv(OUTPUT_DIR/"17_mri_only_binary_predictions.csv",index=False)
display(predictions.head())

Fold model 0 complete | studies=3 | checkpoint=/kaggle/input/notebooks/richashanbhag/13-weak-supervision/13_weak_supervision/gold_plus_confidence_weighted_weak_fold_0.pth
Fold model 1 complete | studies=3 | checkpoint=/kaggle/input/notebooks/richashanbhag/13-weak-supervision/13_weak_supervision/gold_plus_confidence_weighted_weak_fold_1.pth
Fold model 2 complete | studies=3 | checkpoint=/kaggle/input/notebooks/richashanbhag/13-weak-supervision/13_weak_supervision/gold_plus_confidence_weighted_weak_fold_2.pth
Fold model 3 complete | studies=3 | checkpoint=/kaggle/input/notebooks/richashanbhag/13-weak-supervision/13_weak_supervision/gold_plus_confidence_weighted_weak_fold_3.pth
Fold model 4 complete | studies=3 | checkpoint=/kaggle/input/notebooks/richashanbhag/13-weak-supervision/13_weak_supervision/gold_plus_confidence_weighted_weak_fold_4.pth


,StudyInstanceUID,ACL_prob,MCL_prob,Medial Meniscus_prob,Lateral Meniscus_prob,Medial OA_prob,Lateral OA_prob,PF OA_prob,Effusion_prob,Synovitis_prob,Baker's_prob,Contusion_prob,Fracture_prob
0,1.2.826.0.1.3680043.8.498.10047035057544427318...,0.670345,0.415066,0.861946,0.805460,0.211695,0.211319,0.351476,0.836815,0.711875,0.430317,0.513029,0.426561
1,1.2.826.0.1.3680043.8.498.10062861783145312629...,0.634248,0.536932,0.635788,0.599997,0.183292,0.146581,0.261481,0.688044,0.625087,0.189449,0.595647,0.503045
2,1.2.826.0.1.3680043.8.498.10067514707072572280...,0.631878,0.327694,0.737672,0.598396,0.234487,0.142086,0.353338,0.765197,0.502217,0.284636,0.536608,0.501554


## 6. Align predictions to the competition submission template when possible

The notebook creates a standard study-level probability CSV regardless. A `sample_submission.csv` is only populated automatically when it explicitly contains `StudyInstanceUID` and all 12 finding columns. Other submission schemas require a documented mapping; the notebook will not guess how to reshape or order labels.

In [7]:
submission_created=False
if SAMPLE_SUB_PATH:
    sample=pd.read_csv(SAMPLE_SUB_PATH)
    if "StudyInstanceUID" in sample.columns and set(LABEL_COLS).issubset(sample.columns):
        aligned=sample[["StudyInstanceUID"]].copy()
        aligned["StudyInstanceUID"]=aligned["StudyInstanceUID"].astype(str)
        pred_index=predictions.set_index("StudyInstanceUID")
        missing=set(aligned["StudyInstanceUID"])-set(pred_index.index)
        if missing: raise ValueError(f"Sample submission contains {len(missing)} studies without predictions.")
        for label in LABEL_COLS: aligned[label]=aligned["StudyInstanceUID"].map(pred_index[f"{label}_prob"])
        aligned.to_csv(OUTPUT_DIR/"17_submission.csv",index=False)
        submission_created=True
        print("Created 17_submission.csv using explicit StudyInstanceUID + 12-column template alignment.")
    else:
        print("sample_submission.csv was found, but its schema is not an explicit StudyInstanceUID + 12-label matrix.")
        print("No guessed submission reshaping was performed. Use 17_mri_only_predictions.csv as the source for the competition-specific mapping.")
else:
    print("sample_submission.csv not attached. Study-level MRI-only predictions have still been exported.")

Created 17_submission.csv using explicit StudyInstanceUID + 12-column template alignment.


## 7. Inference audit and final output checklist

This records the checkpoint family, test embedding files, number of predictions and whether a submission template could be aligned. No reports or target labels are read by this notebook.

In [9]:
expected = ["17_mri_only_predictions.csv", "17_mri_only_binary_predictions.csv"] + [
    f"17_fold_{f}_predictions.csv" for f in range(N_FOLDS)
]

if submission_created:
    expected.append("17_submission.csv")

missing = [
    name for name in expected
    if not (OUTPUT_DIR / name).exists()
]

audit = {
    "notebook": "17_final_inference.ipynb",
    "model_source": MODEL_SOURCE,
    "weak_supervision_mode": WEAK_MODE if MODEL_SOURCE == "notebook13_weighted" else None,
    "input_modality": "MRI-derived DINOv2 embeddings only",
    "reports_used": False,
    "ground_truth_labels_used": False,
    "test_studies": len(TEST_STUDY_IDS),
    "test_embedding_rows": int(len(test_meta)),
    "embedding_dim": EMBED_DIM,
    "fold_models": N_FOLDS,
    "ensemble": "arithmetic mean of five fold-model sigmoid probabilities",
    "threshold_for_binary_export": THRESHOLD,
    "test_embedding_path": TEST_EMBED_PATH,
    "test_metadata_path": TEST_META_PATH,
    "checkpoint_paths": CHECKPOINT_PATHS,
    "sample_submission_found": SAMPLE_SUB_PATH is not None,
    "sample_submission_created": submission_created,
    "missing_artifacts": missing
}

# Convert Path objects and nested structures into JSON-compatible values
def make_json_serializable(obj):
    if isinstance(obj, Path):
        return str(obj)
    elif isinstance(obj, dict):
        return {
            str(k): make_json_serializable(v)
            for k, v in obj.items()
        }
    elif isinstance(obj, (list, tuple)):
        return [
            make_json_serializable(v)
            for v in obj
        ]
    else:
        return obj

audit = make_json_serializable(audit)

# Save inference manifest
manifest_path = OUTPUT_DIR / "17_inference_manifest.json"

with open(manifest_path, "w") as f:
    json.dump(audit, f, indent=2)

print(json.dumps(audit, indent=2))

# Validate expected output artifacts
if missing:
    raise RuntimeError(
        "Notebook 17 missing expected outputs: " + ", ".join(missing)
    )

print("Inference manifest saved:", manifest_path)
print("NOTEBOOK 17 COMPLETED — MRI-only inference.")

{
  "notebook": "17_final_inference.ipynb",
  "model_source": "notebook13_weighted",
  "weak_supervision_mode": "gold_plus_confidence_weighted_weak",
  "input_modality": "MRI-derived DINOv2 embeddings only",
  "reports_used": false,
  "ground_truth_labels_used": false,
  "test_studies": 3,
  "test_embedding_rows": 557,
  "embedding_dim": 384,
  "fold_models": 5,
  "ensemble": "arithmetic mean of five fold-model sigmoid probabilities",
  "threshold_for_binary_export": 0.5,
  "test_embedding_path": "/kaggle/working/17_final_inference/dinov2_vits14_test_embeddings_float16.npy",
  "test_metadata_path": "/kaggle/working/17_final_inference/dinov2_vits14_test_embedding_metadata.csv",
  "checkpoint_paths": {
    "0": "/kaggle/input/notebooks/richashanbhag/13-weak-supervision/13_weak_supervision/gold_plus_confidence_weighted_weak_fold_0.pth",
    "1": "/kaggle/input/notebooks/richashanbhag/13-weak-supervision/13_weak_supervision/gold_plus_confidence_weighted_weak_fold_1.pth",
    "2": "/kaggle/